# WELFake Dataset – Supervised Text Classification

This notebook continues from the EDA & Data Preparation notebook (which applied **undersampling** to give an exact 50/50 class balance) and covers:

**Section 3 – Supervised Text Classification**
- 3.2 TF-IDF Vectorisation
- 3.3 Multinomial Naive Bayes (baseline)
- 3.4 Logistic Regression (baseline)
- 3.5 Linear SVM (baseline)
- 3.6 Decision Tree (baseline)
- Baseline evaluation using Accuracy, Precision, Recall, F1-score

**Section 4 – Hyperparameter Selection**
- 4.1–4.4 Hyperparameters of each model
- 4.5 Grid Search Implementation
- 4.6 Best Hyperparameters
- Tuned-model evaluation and comparison against the baseline


## Setup

We import the libraries required for vectorisation, model building, hyperparameter tuning, and evaluation.

In [ ]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, classification_report, confusion_matrix)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
RANDOM_STATE = 42


## 3.1 Recap: Load Preprocessed Data

This notebook continues directly from the **EDA & Data Preparation** notebook. Rather than repeating the raw-text cleaning, tokenisation, punctuation removal, and stopword removal here, we load the already-preprocessed `WELFake_processed.csv` file (containing `final_text` and `label`) that was saved at the end of that notebook. That notebook also applied **undersampling**, so this dataset is already class-balanced (50/50).

> **Important:** Run the EDA & Data Preparation notebook first so that `WELFake_processed.csv` exists in the same folder as this notebook.

In [ ]:
df = pd.read_csv('WELFake_processed.csv')

# Drop any rows that became empty after cleaning/stopword removal (e.g. very short titles/text)
df = df.dropna(subset=['final_text']).reset_index(drop=True)
df = df[df['final_text'].str.strip() != ''].reset_index(drop=True)

print(f"Loaded {df.shape[0]} preprocessed articles from 'WELFake_processed.csv'")
print("\nClass distribution (should be ~50/50, confirming the undersampling carried through):")
print(df['label'].value_counts())
df.head()

In [ ]:
X = df['final_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set: {X_train.shape[0]} articles")
print(f"Testing set:  {X_test.shape[0]} articles")

## 3.2 TF-IDF Vectorisation

This applies the **Lab 10** concept of converting raw text into numerical feature vectors. Unlike a simple Count Vectorizer (which only counts word occurrences), **TF-IDF (Term Frequency – Inverse Document Frequency)** down-weights words that occur frequently across *all* documents (which carry little discriminative value) and up-weights words that are frequent in a specific document but rare across the corpus (which are more informative for classification).

We fit the `TfidfVectorizer` **only on the training data** (to avoid data leakage from the test set) and then use it to transform both the training and testing sets. `max_features` is capped at 5000 to keep the feature space computationally manageable while retaining the most informative terms, and `ngram_range=(1, 2)` includes both unigrams and bigrams to capture short meaningful phrases (e.g. "fake news", "breaking news").

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print(f"TF-IDF training matrix shape: {X_train_tfidf.shape}")
print(f"TF-IDF testing matrix shape:  {X_test_tfidf.shape}")
print(f"\nSample vocabulary terms: {list(tfidf_vectorizer.vocabulary_.keys())[:15]}")

### Evaluation Helper Function

To keep the evaluation consistent across all models, we define a helper function that computes **Accuracy, Precision, Recall, and F1-score** (using `average='weighted'`), prints a classification report, and plots a confusion matrix.

In [ ]:
results = {}  # stores baseline results for later comparison
tuned_results = {}  # stores tuned results for later comparison

def evaluate_model(name, model, X_te, y_te, store_in=None):
    y_pred = model.predict(X_te)

    acc = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, average='weighted')
    rec = recall_score(y_te, y_pred, average='weighted')
    f1 = f1_score(y_te, y_pred, average='weighted')

    print(f"===== {name} =====")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")
    print("\nClassification Report:\n", classification_report(y_te, y_pred, target_names=['Fake (0)', 'Real (1)']))

    cm = confusion_matrix(y_te, y_pred)
    plt.figure(figsize=(4, 3.5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Fake', 'Real'], yticklabels=['Fake', 'Real'])
    plt.title(f'Confusion Matrix – {name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

    metrics = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1-score': f1}
    if store_in is not None:
        store_in[name] = metrics
    return metrics

## 3.3 Multinomial Naive Bayes (Baseline)

Multinomial Naive Bayes is well-suited to text classification with word-frequency-style features (like TF-IDF or counts) as it assumes conditional independence between features given the class, which is a strong but computationally cheap and often effective assumption for text data.

In [ ]:
nb_baseline = MultinomialNB()
nb_baseline.fit(X_train_tfidf, y_train)

evaluate_model('Naive Bayes (Baseline)', nb_baseline, X_test_tfidf, y_test, store_in=results)

## 3.4 Logistic Regression (Baseline)

Logistic Regression is a linear model that estimates the probability of an article being real vs. fake based on a weighted combination of its TF-IDF features. `class_weight='balanced'` is kept for robustness, although since the training data is now already balanced 50/50 by the undersampling step, its effect here is minimal.

In [ ]:
lr_baseline = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)
lr_baseline.fit(X_train_tfidf, y_train)

evaluate_model('Logistic Regression (Baseline)', lr_baseline, X_test_tfidf, y_test, store_in=results)

## 3.5 Linear SVM (Baseline)

A Linear Support Vector Machine finds the hyperplane that best separates the two classes with the maximum margin. `LinearSVC` is used (rather than `SVC` with a linear kernel) because it scales far better to high-dimensional, sparse text data such as TF-IDF matrices, which is standard practice for text classification.

In [ ]:
svm_baseline = LinearSVC(class_weight='balanced', random_state=RANDOM_STATE)
svm_baseline.fit(X_train_tfidf, y_train)

evaluate_model('Linear SVM (Baseline)', svm_baseline, X_test_tfidf, y_test, store_in=results)

## 3.6 Decision Tree (Baseline)

A Decision Tree splits the feature space into regions using a sequence of if/else rules learned from the training data, choosing splits that best separate the classes at each step. It is simple and highly interpretable (the learned rules can be inspected directly), but a single unconstrained tree can easily overfit high-dimensional TF-IDF features by memorising very specific word patterns -- which is exactly what the hyperparameter tuning in Section 4 is aimed at controlling.


In [ ]:
dt_baseline = DecisionTreeClassifier(
    class_weight='balanced',
    random_state=RANDOM_STATE
)
dt_baseline.fit(X_train_tfidf, y_train)

evaluate_model('Decision Tree (Baseline)', dt_baseline, X_test_tfidf, y_test, store_in=results)


### Baseline Model Comparison

We now compare all four baseline models side by side on the four evaluation metrics. The y-axis is zoomed to the range the scores actually occupy (rather than a fixed 0–1 scale), and each bar is labelled with its exact value, so the differences between models are easy to read.

In [ ]:
results_df = pd.DataFrame(results).T
print(results_df.round(4))

metrics_list = ['Accuracy', 'Precision', 'Recall', 'F1-score']
model_colors = {
    'Naive Bayes (Baseline)': '#4e79a7',
    'Logistic Regression (Baseline)': '#f28e2b',
    'Linear SVM (Baseline)': '#59a14f',
    'Decision Tree (Baseline)': '#e15759',
}

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for ax, metric in zip(axes.flatten(), metrics_list):
    vals = results_df[metric]
    colors = [model_colors.get(m, 'gray') for m in vals.index]
    bars = ax.bar(vals.index, vals.values, color=colors, edgecolor='black', linewidth=0.5, width=0.6)

    # Zoom each metric's y-axis to its OWN score range (not a shared range
    # across all four metrics) -- this is what makes small differences
    # between models actually visible.
    v_min, v_max = vals.min(), vals.max()
    pad = (v_max - v_min) * 0.4 if v_max > v_min else 0.02
    ax.set_ylim(max(0, v_min - pad), min(1.0, v_max + pad))
    ax.set_title(metric)
    ax.set_xticklabels(vals.index, rotation=15, ha='right')
    ax.grid(axis='y', alpha=0.3)
    ax.bar_label(bars, fmt='%.3f', fontsize=8, padding=2)

plt.tight_layout()
plt.suptitle('Baseline Model Comparison (each metric individually zoomed for readability)', y=1.02, fontsize=13)
plt.show()


## 4.0 Hyperparameter Selection

Each baseline model above was trained with mostly default settings. This section identifies the key hyperparameters of each model, tunes them using **Grid Search with cross-validation**, and evaluates whether tuning improves performance over the baseline.

> **Note on computation:** Grid search over the full training set for every parameter combination is computationally expensive as the parameter grid grows. To keep runtime practical, grid search is performed using **3-fold cross-validation** with focused, small parameter grids. This is a standard trade-off in applied machine learning between exhaustive search and available compute time.


### 4.1 Naive Bayes Hyperparameters

| Hyperparameter | Description |
|---|---|
| `alpha` | Additive (Laplace/Lidstone) smoothing parameter. Prevents zero probabilities for words unseen in a class during training. Smaller values fit the training data more closely (risk of overfitting); larger values smooth predictions more (risk of underfitting). |
| `fit_prior` | Whether to learn class prior probabilities from the data (`True`) or assume uniform priors (`False`). |


### 4.2 Logistic Regression Hyperparameters

| Hyperparameter | Description |
|---|---|
| `C` | Inverse of regularisation strength. Smaller `C` = stronger regularisation (simpler model, less overfitting); larger `C` = weaker regularisation (fits training data more closely). |
| `penalty` | Type of regularisation applied to the model coefficients (`l1` for sparse/feature-selecting solutions, `l2` for shrinking coefficients smoothly). |
| `solver` | The optimisation algorithm used to fit the model (e.g. `liblinear`, which supports both `l1` and `l2` penalties and works well for smaller/medium datasets). |


### 4.3 Linear SVM Hyperparameters

| Hyperparameter | Description |
|---|---|
| `C` | Regularisation parameter controlling the trade-off between maximising the margin and minimising classification error on the training data. Smaller `C` allows more margin violations (softer margin); larger `C` fits the training data more strictly. |
| `loss` | The loss function used (`hinge` is the standard SVM loss; `squared_hinge` penalises margin violations quadratically, making it more sensitive to outliers). |


### 4.4 Decision Tree Hyperparameters

| Hyperparameter | Description |
|---|---|
| `max_depth` | Maximum depth of the tree. Unrestricted depth (`None`) tends to overfit high-dimensional TF-IDF data by learning very specific word patterns; shallower trees generalise better but may underfit. |
| `min_samples_split` | Minimum number of samples required to split an internal node. Higher values produce simpler, less overfit trees. |
| `min_samples_leaf` | Minimum number of samples required at a leaf node. Higher values smooth the model and help prevent overfitting to rare word patterns. |
| `criterion` | The function used to measure split quality (`gini` impurity or `entropy`/information gain). |


## 4.5 Grid Search Implementation

For each model, `GridSearchCV` exhaustively searches over the specified parameter grid using **Stratified 3-Fold Cross-Validation** (to preserve class balance in every fold) and selects the combination that maximises the weighted F1-score.

In [ ]:
cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

grid_search_results = {}
best_models = {}

def run_grid_search(name, estimator, param_grid):
    start = time.time()
    grid = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        scoring='f1_weighted',
        cv=cv_strategy,
        n_jobs=-1,
        verbose=0
    )
    grid.fit(X_train_tfidf, y_train)
    elapsed = time.time() - start

    print(f"===== {name} Grid Search =====")
    print(f"Best CV F1-score: {grid.best_score_:.4f}")
    print(f"Best Parameters : {grid.best_params_}")
    print(f"Time taken      : {elapsed:.1f} seconds\n")

    grid_search_results[name] = grid
    best_models[name] = grid.best_estimator_
    return grid

In [ ]:
# --- Naive Bayes ---
nb_param_grid = {
    'alpha': [0.01, 0.1, 0.5, 1.0, 2.0],
    'fit_prior': [True, False]
}
nb_grid = run_grid_search('Naive Bayes', MultinomialNB(), nb_param_grid)

In [ ]:
# --- Logistic Regression ---
lr_param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l2'],
    'solver': ['liblinear']
}
lr_grid = run_grid_search('Logistic Regression',
                           LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE),
                           lr_param_grid)

In [ ]:
# --- Linear SVM ---
svm_param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'loss': ['hinge', 'squared_hinge']
}
svm_grid = run_grid_search('Linear SVM',
                            LinearSVC(class_weight='balanced', random_state=RANDOM_STATE),
                            svm_param_grid)

In [ ]:
# --- Decision Tree ---
dt_param_grid = {
    'max_depth': [10, 30, 50, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5],
    'criterion': ['gini', 'entropy']
}
dt_grid = run_grid_search('Decision Tree',
                           DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_STATE),
                           dt_param_grid)


## 4.6 Best Hyperparameters

Summary of the best hyperparameter combination found for each model, based on cross-validated weighted F1-score.

In [ ]:
best_params_summary = pd.DataFrame({
    'Model': list(grid_search_results.keys()),
    'Best Parameters': [g.best_params_ for g in grid_search_results.values()],
    'Best CV F1-score': [round(g.best_score_, 4) for g in grid_search_results.values()]
})
best_params_summary

## Evaluation of Tuned Models on the Test Set

We now evaluate the best model found for each algorithm on the held-out **test set**, using the same Accuracy, Precision, Recall, and F1-score metrics as the baseline evaluation.

In [ ]:
evaluate_model('Naive Bayes (Tuned)', best_models['Naive Bayes'], X_test_tfidf, y_test, store_in=tuned_results)

In [ ]:
evaluate_model('Logistic Regression (Tuned)', best_models['Logistic Regression'], X_test_tfidf, y_test, store_in=tuned_results)

In [ ]:
evaluate_model('Linear SVM (Tuned)', best_models['Linear SVM'], X_test_tfidf, y_test, store_in=tuned_results)

In [ ]:
evaluate_model('Decision Tree (Tuned)', best_models['Decision Tree'], X_test_tfidf, y_test, store_in=tuned_results)


## Baseline vs Tuned: Final Comparison

Finally, we compare each model's baseline (default hyperparameters) performance against its tuned (grid-searched) performance, to see whether hyperparameter tuning provided a meaningful improvement. As with the baseline comparison chart, each subplot's y-axis is zoomed to the range the scores actually occupy, and every bar is labelled with its exact value.

In [ ]:
comparison_rows = []
model_names = ['Naive Bayes', 'Logistic Regression', 'Linear SVM', 'Decision Tree']

for m in model_names:
    baseline_key = [k for k in results if m.split(' ')[0] in k or m in k][0]
    tuned_key = f'{m} (Tuned)'
    for metric in ['Accuracy', 'Precision', 'Recall', 'F1-score']:
        comparison_rows.append({
            'Model': m,
            'Metric': metric,
            'Baseline': results[baseline_key][metric],
            'Tuned': tuned_results[tuned_key][metric]
        })

comparison_df = pd.DataFrame(comparison_rows)
comparison_pivot = comparison_df.pivot(index='Model', columns='Metric', values=['Baseline', 'Tuned'])
print(comparison_pivot.round(4))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
metrics_list = ['Accuracy', 'Precision', 'Recall', 'F1-score']

for ax, metric in zip(axes.flatten(), metrics_list):
    subset = comparison_df[comparison_df['Metric'] == metric]
    x = np.arange(len(subset['Model']))
    width = 0.35

    bars1 = ax.bar(x - width/2, subset['Baseline'], width, label='Baseline', color='steelblue', edgecolor='black', linewidth=0.5)
    bars2 = ax.bar(x + width/2, subset['Tuned'], width, label='Tuned', color='darkorange', edgecolor='black', linewidth=0.5)

    ax.set_xticks(x)
    ax.set_xticklabels(subset['Model'], rotation=15)
    ax.set_title(metric)

    all_vals = pd.concat([subset['Baseline'], subset['Tuned']])
    v_min, v_max = all_vals.min(), all_vals.max()
    # Extra headroom (vs. the baseline chart) to leave room for the delta
    # labels placed above each pair of bars.
    pad = (v_max - v_min) * 0.5 if v_max > v_min else 0.03
    ax.set_ylim(max(0, v_min - pad * 0.4), min(1.0, v_max + pad))
    ax.grid(axis='y', alpha=0.3)

    ax.bar_label(bars1, fmt='%.3f', fontsize=7, padding=2)
    ax.bar_label(bars2, fmt='%.3f', fontsize=7, padding=2)

    # Delta annotation: makes the direction/size of the tuning effect
    # immediately readable instead of having to eyeball bar heights.
    baseline_vals = subset['Baseline'].values
    tuned_vals = subset['Tuned'].values
    deltas = tuned_vals - baseline_vals
    top_of_pair = np.maximum(baseline_vals, tuned_vals)
    label_y = top_of_pair + pad * 0.55
    for xi, dy, d in zip(x, label_y, deltas):
        color = '#2ca02c' if d >= 0 else '#d62728'
        arrow = '\u25b2' if d >= 0 else '\u25bc'
        ax.text(xi, dy, f'{arrow} {d:+.3f}', ha='center', va='bottom',
                fontsize=8, fontweight='bold', color=color)

    ax.legend(fontsize=8, loc='lower left')

plt.tight_layout()
plt.suptitle('Baseline vs Tuned Model Performance (\u25b2/\u25bc labels show the tuning delta)', y=1.02, fontsize=14)
plt.show()


### Summary

- The **baseline** models used default (or near-default) hyperparameters, giving an initial sense of each algorithm's out-of-the-box performance on the TF-IDF features, extracted from a dataset that was **undersampled to an exact 50/50 class balance** during data preparation.
- **Grid Search with cross-validation** was used to systematically search each model's hyperparameter space and select the combination maximising weighted F1-score on the training data, including for **Decision Tree**, which replaced XGBoost as the fourth model: since a single tree is prone to overfitting high-dimensional TF-IDF features, the grid focuses on the pruning-related hyperparameters (`max_depth`, `min_samples_split`, `min_samples_leaf`) most likely to control that.
- The **tuned models** are then evaluated on the same held-out test set as the baselines, allowing a fair before/after comparison per model. Comparison charts throughout the notebook use per-metric zoomed axes, gridlines, and value/delta labels so that differences between models (and between baseline and tuned versions of the same model) are easy to read at a glance rather than compressed onto a single 0-1 scale.
- These results, together with the confusion matrices, should be used in the report's evaluation and discussion section (Q4) to critically compare the models and justify which one is best suited to the fake-news detection task.
